In [4]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="/Users/satvikpandey/Ai_project/dataset.jsonl.jsonl")
dataset = dataset["train"].train_test_split(test_size=0.1)

Step-1

Imports HuggingFace datasets library

→ used to load and manage training data.

Step-2

Loads your dataset from JSONL file.

Your dataset contains:

input → question
output → answer

So this becomes a structured dataset.

Step-3

Splits dataset:

90% → training
10% → testing

This is important because:

👉 Model must be evaluated on unseen data.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "mps"

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16
)

model.to(device)

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

Cell 2 — Loading the Base Pretrained Model

In this step, the required deep learning libraries such as PyTorch and Hugging Face Transformers are imported.
The base Large Language Model (LLM), such as TinyLlama-1.1B-Chat, is then loaded along with its tokenizer.

The tokenizer converts human-readable text into numerical tokens that the model can process.
The model is moved to the Apple Silicon GPU (MPS device) to accelerate training and reduce computation time.

In [2]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

Cell 3 — Applying LoRA for Parameter-Efficient Fine-Tuning

This cell configures LoRA (Low-Rank Adaptation), which is a parameter-efficient fine-tuning technique.
Instead of updating all model weights, LoRA introduces small trainable matrices in selected attention layers.

This approach significantly reduces:

GPU memory usage
Training time
Computational cost

Only a small percentage of parameters (typically 1–2%) are updated during training, making it suitable for limited hardware environments.

In [5]:
def format_prompt(ex):
    return {
        "text": f"User: {ex['input']}\nAssistant: {ex['output']}"
    }

dataset = dataset.map(format_prompt)

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Cell 4 — Formatting Training Prompts

Here, the dataset samples are converted into a chat-style prompt format.
Each question-answer pair is transformed into a structured conversational template:

User: <question>
Assistant: <answer>

This formatting teaches the model to behave like a chatbot and generate responses in a conversational style during inference.

In [6]:
def tokenize(ex):
    return tokenizer(
        ex["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Cell 5 — Tokenization of the Dataset

This cell converts formatted text into token IDs using the tokenizer.
Tokenization ensures that all inputs have a fixed maximum sequence length by applying:

truncation for long text
padding for shorter text

This step is necessary because neural networks require numerical inputs of consistent size.

In [7]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="mac_drone_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10
)

Cell 6 — Defining Training Configuration

Training hyperparameters are defined using the TrainingArguments class.
Key configurations include:

output directory for saving checkpoints
batch size and gradient accumulation
number of training epochs
learning rate for optimization
logging frequency

These parameters control the speed, stability, and effectiveness of the fine-tuning process.

In [8]:
from transformers import Trainer, DataCollatorForLanguageModeling

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

/Users/satvikpandey/Ai_project/ai_env/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,2.658106
20,2.432263
30,2.195401
40,2.126404
50,2.017989
60,2.017280
70,1.932869
80,1.973466
90,1.926528
100,1.927494


TrainOutput(global_step=171, training_loss=2.02097141115289, metrics={'train_runtime': 1485.5144, 'train_samples_per_second': 0.909, 'train_steps_per_second': 0.115, 'total_flos': 4295001165004800.0, 'train_loss': 2.02097141115289, 'epoch': 3.0})

Cell 7 — Trainer Initialization and Model Training

The Hugging Face Trainer API is used to simplify the training loop.
This component connects:

the model
training configuration
tokenized datasets
data collator for batching

When training begins, the model learns domain-specific knowledge from the dataset and adapts its internal representations accordingly.

In [11]:
model.save_pretrained("mac_drone_model")
tokenizer.save_pretrained("mac_drone_model")

/Users/satvikpandey/Ai_project/ai_env/lib/python3.14/site-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error _ssl.c:1059: The handshake operation timed out - silently ignoring the lookup for the file config.json in TinyLlama/TinyLlama-1.1B-Chat-v1.0.
  warnings.warn(
/Users/satvikpandey/Ai_project/ai_env/lib/python3.14/site-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in TinyLlama/TinyLlama-1.1B-Chat-v1.0 - will assume that the vocabulary was not modified.
  warnings.warn(


('mac_drone_model/tokenizer_config.json',
 'mac_drone_model/chat_template.jinja',
 'mac_drone_model/tokenizer.json')

Cell 8 — Saving the Fine-Tuned Model (LoRA Adapter)

After training, the fine-tuned model and tokenizer are saved locally.
At this stage, the saved model still depends on the LoRA adapter, meaning the base model and adapter must be loaded together for inference.

This checkpoint is useful for continuing training or performing further experimentation.

In [9]:
from peft import PeftModel

merged_model = model.merge_and_unload()

merged_model.save_pretrained("mac_drone_model_merged")
tokenizer.save_pretrained("mac_drone_model_merged")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('mac_drone_model_merged/tokenizer_config.json',
 'mac_drone_model_merged/chat_template.jinja',
 'mac_drone_model_merged/tokenizer.json')

Cell 9 — Merging LoRA Weights into the Base Model

This step merges the trained LoRA parameters directly into the base model weights.
The resulting model becomes a standalone deployable model, eliminating the need for PEFT libraries during inference.

This merged model is more convenient for:

deployment
sharing
integration into applications

In [10]:
prompt = """You are Wildlife Drone Assistant.
User: What are system limitations?
Assistant:"""

Cell 10 — Testing the Model with a Sample Prompt

A sample prompt is defined to evaluate how the model responds after fine-tuning.
This helps verify whether the model has successfully learned project-specific concepts and can generate coherent technical answers.

Such testing is important for qualitative evaluation before deployment.